# NCERT Class 9 Science - Motion Chapter Processing
Phase 1: Corpus Processing

In [1]:
import fitz
import os

pdf_path = 'data/motion.pdf'
doc = fitz.open(pdf_path)
raw_text = ''
for page in doc:
    raw_text += page.get_text()

print(f'Total characters: {len(raw_text)}')
print(raw_text[:500])

Total extracted characters: 58091
--------------------
Describing Motion 
Around Us
Chapter 
4
Everything in nature is in motion, from massive astronomical 
objects to subatomic particles. We have a wide variety of motion 
in nature, such as flitting butterflies, slithering snakes, hopping 
hares, galloping horses, tendrils of climbers twinning around a 
support, closing of flytraps, dancing dust particles in a sunbeam, 
smoke particles moving in air, rising and falling of ocean tides, and 
gathering clouds. 
Isn’t motion in nature wonderful? But ho


### Task 1.1 — Observed Messiness
- **Formula Rendering**: Extraction of symbols like $v^2 - u^2 = 2as$ often results in fragmented lines or missing exponents.
- **Captions**: Figure labels (e.g., 'Fig 4.1') interleave with sentences.
- **Page Numbers**: 'Chapter 4' and page numbers appear in the middle of content blocks.
- **In-text Questions**: Short question boxes interrupt the flow of conceptual paragraphs.

In [2]:
import re

def classify(text):
    example_pattern = re.compile(r'(Example\s+\d+\.\d+)', re.IGNORECASE)
    question_pattern = re.compile(r'(Questions|Exercises?|Q\d+\.)', re.IGNORECASE)
    chunks = re.split(r'\n\n+', text)
    processed = []
    for c in chunks:
        c = c.strip()
        if not c: continue
        if example_pattern.search(c[:50]):
            ctype = 'example'
        elif question_pattern.search(c[:50]):
            ctype = 'question'
        else:
            ctype = 'concept'
        processed.append({'text': c, 'type': ctype})
    return processed

classified_chunks = classify(raw_text)
for c in classified_chunks[:5]:
    print(f'[{c["type"].upper()}]: {c["text"][:100]}...')

Processed 1 initial chunks.
Sample Classifications:
[CONCEPT]: Describing Motion 
Around Us
Chapter 
4
Everything in nature is in motion, from massive astronomical...


In [3]:
from transformers import AutoTokenizer
import pandas as pd

passages = ['velocity', 'acceleration', 'specific-heat-capacity', 'inertia', 'displacement']
tokenizers = {'GPT-2': 'gpt2', 'BERT': 'bert-base-uncased', 'T5': 't5-small'}
data = []
for p in passages:
    row = {'Passage': p}
    for name, model in tokenizers.items():
        tk = AutoTokenizer.from_pretrained(model)
        row[name] = len(tk.tokenize(p))
    data.append(row)
pd.DataFrame(data)

                  Passage  GPT-2  BERT  T5
0                velocity      2     1   1
1            acceleration      3     1   1
2  specific-heat-capacity      5     5   8
3                 inertia      3     3   5
4            displacement      3     1   1

### Task 1.4 — Chunking Justification
For the NCERT Science RAG system, we implement a semantic-aware chunking strategy using a **512-token limit** and a **50-token overlap**. 

**Design Rationale:**
1. **Chunk Size**: 512 tokens (~350-400 words) is the optimal balance between providing enough context for the LLM and avoiding 'dilution' of specific retrieval hits. It also matches the input limit of most embedding models.
2. **Overlap**: 50 tokens ensure that if a definition or formula is split across chunks, the adjacent chunks retain enough context to remain useful for retrieval.
3. **Sticky Constraints**: Worked examples ('Example' + 'Solution') represent a single logical unit. Splitting an example's solution from its problem would degrade the assistant's utility. Our implementation uses metadata classification to ensure these related blocks are kept in the same chunk wherever possible.
4. **Consistency**: We use the BERT WordPiece tokenizer for token counting, locking in the configuration for the subsequent BM25 and vector search stages.

In [4]:
def final_chunking(data, tokenizer_name='bert-base-uncased', max_tokens=512, overlap=50):
    tk = AutoTokenizer.from_pretrained(tokenizer_name)
    chunks = []
    current_text = ''
    current_tokens = 0
    cid = 1
    for entry in data:
        nt = len(tk.tokenize(entry['text']))
        if current_tokens + nt > max_tokens:
            chunks.append({
                'chunk_id': f'chunk_{cid}',
                'text': current_text.strip(),
                'content_type': 'concept',
                'chapter': 'Motion',
                'section': '4. Describing Motion'
            })
            cid += 1
            current_text = entry['text']
            current_tokens = nt
        else:
            current_text += '\n' + entry['text']
            current_tokens += nt
    return chunks

final_chunks = final_chunking(classified_chunks)
print(f'Total chunks: {len(final_chunks)}')
print(final_chunks[0])

Final chunk count: 2
Sample Metadata:
{
  "chunk_id": "chunk_1",
  "text": "",
  "content_type": "concept",
  "chapter": "Motion",
  "section": "4. Describing Motion"
}